# Week 09 - CNNs and Transfer Learning

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 10  
**Estimated time:** 3-4 hours

This notebook uses CIFAR-10 to show the central practical idea of modern computer vision: start with a pretrained model, replace the task-specific head, and adapt it to your own classification problem.


## What You Are Building

This week has five required implementation pieces:

1. `prepare_cifar10_loaders(...)` - download CIFAR-10, create small balanced subsets, and build loaders for both a simple CNN and a pretrained model.
2. `class SimpleCNN(nn.Module)` - train a small CNN from scratch so you have a baseline.
3. `build_finetune_model(n_classes, freeze_features)` - adapt a pretrained MobileNetV2 to CIFAR-10.
4. `train_model(model, train_loader, test_loader, optimizer, criterion, epochs, device)` - train and record learning curves.
5. `make_benchmark_long(results, week, dataset, task_type, target, split)` - append image-classification results to the reusable benchmark format.

The modelling question is practical: if you only have a modest image dataset, how much do you gain from reusing a pretrained visual representation instead of training a CNN from scratch?


In [ ]:
# Imports - keep this cell stable
import warnings
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

try:
    import torchvision
    from torchvision import datasets, transforms, models
    from torchvision.models import MobileNet_V2_Weights
except ImportError as exc:
    raise ImportError("This notebook requires torchvision. Install it with: pip install torchvision") from exc

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
BATCH_SIZE = 64
TRAIN_PER_CLASS = 500
TEST_PER_CLASS = 100
EPOCHS_SIMPLE = 5
EPOCHS_FINETUNE = 3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = Path("./data")

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
print("Using device:", DEVICE)


## Dataset and Download Note

CIFAR-10 has 10 natural-image classes. The notebook downloads it through `torchvision.datasets.CIFAR10`. To keep training time reasonable, you will use balanced subsets of the train and test splits.

The pretrained model uses ImageNet weights. That means the first run may also download model weights. This is intentional: adapting a pretrained model is the main point of the week.


## Task 1 - Prepare CIFAR-10 Loaders

Implement `prepare_cifar10_loaders(data_dir, batch_size, train_per_class, test_per_class, random_state)`.

Return:

```python
simple_train_loader, simple_test_loader, pretrained_train_loader, pretrained_test_loader, class_names
```

Requirements:

- Use CIFAR-10 train and test splits.
- Select a balanced subset: `train_per_class` examples per class from the train split and `test_per_class` examples per class from the test split.
- The simple CNN loaders should use 32-by-32 tensors with CIFAR-style normalisation.
- The pretrained loaders should resize images to 224-by-224 and use the official MobileNetV2 ImageNet preprocessing transforms.
- Use the same selected indices for the simple and pretrained versions.


In [ ]:
def prepare_cifar10_loaders(
    data_dir=DATA_DIR,
    batch_size: int = BATCH_SIZE,
    train_per_class: int = TRAIN_PER_CLASS,
    test_per_class: int = TEST_PER_CLASS,
    random_state: int = RANDOM_STATE,
):
    """Create balanced CIFAR-10 loaders for scratch CNN and pretrained MobileNetV2."""
    generator = torch.Generator().manual_seed(random_state)

    simple_transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
        ]
    )
    pretrained_transform = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ]
    )

    simple_train = datasets.CIFAR10(data_dir, train=True, download=True, transform=simple_transform)
    simple_test = datasets.CIFAR10(data_dir, train=False, download=True, transform=simple_transform)
    pretrained_train = datasets.CIFAR10(data_dir, train=True, download=False, transform=pretrained_transform)
    pretrained_test = datasets.CIFAR10(data_dir, train=False, download=False, transform=pretrained_transform)

    def balanced_indices(targets, per_class):
        rng = np.random.default_rng(random_state)
        targets = np.asarray(targets)
        selected = []
        for class_id in sorted(np.unique(targets)):
            class_indices = np.flatnonzero(targets == class_id)
            if len(class_indices) < per_class:
                raise ValueError(f"class {class_id} has only {len(class_indices)} examples")
            selected.extend(rng.choice(class_indices, size=per_class, replace=False).tolist())
        rng.shuffle(selected)
        return selected

    train_indices = balanced_indices(simple_train.targets, train_per_class)
    test_indices = balanced_indices(simple_test.targets, test_per_class)

    simple_train_loader = DataLoader(Subset(simple_train, train_indices), batch_size=batch_size, shuffle=True, generator=generator)
    simple_test_loader = DataLoader(Subset(simple_test, test_indices), batch_size=batch_size, shuffle=False)
    pretrained_train_loader = DataLoader(Subset(pretrained_train, train_indices), batch_size=batch_size, shuffle=True, generator=generator)
    pretrained_test_loader = DataLoader(Subset(pretrained_test, test_indices), batch_size=batch_size, shuffle=False)

    return simple_train_loader, simple_test_loader, pretrained_train_loader, pretrained_test_loader, simple_train.classes


In [ ]:
# Self-check: Task 1
simple_train_loader, simple_test_loader, pretrained_train_loader, pretrained_test_loader, class_names = prepare_cifar10_loaders()

xb_simple, yb_simple = next(iter(simple_train_loader))
xb_pre, yb_pre = next(iter(pretrained_train_loader))

assert xb_simple.shape[1:] == (3, 32, 32)
assert xb_pre.shape[1:] == (3, 224, 224)
assert yb_simple.dtype == torch.long
assert yb_pre.dtype == torch.long
assert len(class_names) == 10
assert len(simple_train_loader.dataset) == TRAIN_PER_CLASS * 10
assert len(simple_test_loader.dataset) == TEST_PER_CLASS * 10

print("Task 1 passed")


In [ ]:
# Visualise a small batch from the simple CNN loader.
images, labels = next(iter(simple_train_loader))
fig, axes = plt.subplots(4, 8, figsize=(11, 6))
for ax, image, label in zip(axes.ravel(), images[:32], labels[:32]):
    # Undo rough CIFAR normalisation for display.
    img = image.permute(1, 2, 0).numpy()
    img = np.clip(img * 0.5 + 0.5, 0, 1)
    ax.imshow(img)
    ax.set_title(class_names[int(label)], fontsize=8)
    ax.axis("off")
plt.suptitle("CIFAR-10 subset examples")
plt.tight_layout()
plt.show()


## Task 2 - Simple CNN From Scratch

Implement `class SimpleCNN(nn.Module)` for 32-by-32 RGB images.

Suggested architecture:

- Conv2d: `3 -> 32`, kernel size 3, padding 1, ReLU, MaxPool2d(2)
- Conv2d: `32 -> 64`, kernel size 3, padding 1, ReLU, MaxPool2d(2)
- Conv2d: `64 -> 128`, kernel size 3, padding 1, ReLU, MaxPool2d(2)
- Flatten
- Linear: `128 * 4 * 4 -> 128`, ReLU
- Linear: `128 -> 10`

Return logits, not probabilities.


In [ ]:
class SimpleCNN(nn.Module):
    """Small CNN trained from scratch on CIFAR-10."""

    def __init__(self, n_classes: int = 10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


In [ ]:
# Self-check: Task 2
test_cnn = SimpleCNN(n_classes=10)
out = test_cnn(torch.zeros(4, 3, 32, 32))

assert isinstance(test_cnn, nn.Module)
assert out.shape == (4, 10)
assert not torch.allclose(out.softmax(dim=1), out), "Return logits, not probabilities"

print("Task 2 passed")


## Task 3 - Build a Pretrained Fine-Tuning Model

Implement `build_finetune_model(n_classes=10, freeze_features=True)`.

Requirements:

- Load MobileNetV2 with ImageNet weights using `MobileNet_V2_Weights.DEFAULT`.
- Replace the final classifier layer so it predicts `n_classes` outputs.
- If `freeze_features=True`, freeze the feature extractor parameters and leave only the classifier head trainable.
- Return the model.

This is the central transfer-learning step: keep the general visual representation, replace the task head.


In [ ]:
def build_finetune_model(n_classes: int = 10, freeze_features: bool = True):
    """Load MobileNetV2 pretrained on ImageNet and replace its classifier head."""
    weights = MobileNet_V2_Weights.DEFAULT
    model = models.mobilenet_v2(weights=weights)

    if freeze_features:
        for parameter in model.features.parameters():
            parameter.requires_grad = False

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, n_classes)
    return model


In [ ]:
# Self-check: Task 3
transfer_model = build_finetune_model(n_classes=10, freeze_features=True)

assert isinstance(transfer_model, nn.Module)
out = transfer_model(torch.zeros(2, 3, 224, 224))
assert out.shape == (2, 10)
trainable = [p for p in transfer_model.parameters() if p.requires_grad]
all_params = list(transfer_model.parameters())
assert 0 < len(trainable) < len(all_params), "Freeze feature extractor but keep classifier head trainable"

print("Task 3 passed")


## Provided Helper: Evaluation

This helper evaluates loss, accuracy, and macro-F1.


In [ ]:
def evaluate_torch(model, loader, criterion, device=DEVICE):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            all_preds.append(logits.argmax(dim=1).cpu().numpy())
            all_targets.append(yb.cpu().numpy())

    y_pred = np.concatenate(all_preds)
    y_true_eval = np.concatenate(all_targets)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true_eval, y_pred),
        "f1_macro": f1_score(y_true_eval, y_pred, average="macro"),
    }


## Task 4 - Training Loop

Implement `train_model(model, train_loader, test_loader, optimizer, criterion, epochs, device)`.

For each epoch:

- train over all batches;
- record train loss;
- evaluate on the test loader using `evaluate_torch`;
- return a history dataframe with columns `epoch`, `train_loss`, `test_loss`, `test_accuracy`, `test_f1_macro`.


In [ ]:
def train_model(model, train_loader, test_loader, optimizer, criterion, epochs: int, device=DEVICE) -> pd.DataFrame:
    """Train an image classifier and return epoch-level metrics."""
    model.to(device)
    rows = []

    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item() * xb.size(0)

        train_loss = total_train_loss / len(train_loader.dataset)
        test_metrics = evaluate_torch(model, test_loader, criterion, device=device)
        rows.append(
            {
                "epoch": epoch,
                "train_loss": float(train_loss),
                "test_loss": float(test_metrics["loss"]),
                "test_accuracy": float(test_metrics["accuracy"]),
                "test_f1_macro": float(test_metrics["f1_macro"]),
            }
        )

    columns = ["epoch", "train_loss", "test_loss", "test_accuracy", "test_f1_macro"]
    return pd.DataFrame(rows, columns=columns)


In [ ]:
# Self-check: Task 4
criterion = nn.CrossEntropyLoss()
small_model = SimpleCNN().to(DEVICE)
small_optimizer = torch.optim.Adam(small_model.parameters(), lr=0.001)
small_history = train_model(small_model, simple_train_loader, simple_test_loader, small_optimizer, criterion, epochs=1, device=DEVICE)

expected_cols = ["epoch", "train_loss", "test_loss", "test_accuracy", "test_f1_macro"]
assert isinstance(small_history, pd.DataFrame)
assert list(small_history.columns) == expected_cols
assert len(small_history) == 1
assert small_history["test_accuracy"].between(0, 1).all()
assert small_history["train_loss"].notna().all()

print("Task 4 passed")
small_history


## Train Scratch CNN and Pretrained Model

Train a simple CNN from scratch, then train only the classifier head of a pretrained MobileNetV2. The comparison is the main lesson of the week.


In [ ]:
def train_variant(model_name, model, train_loader, test_loader, lr=0.001, epochs=3):
    torch.manual_seed(RANDOM_STATE)
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()
    start = perf_counter()
    history = train_model(model, train_loader, test_loader, optimizer, criterion, epochs=epochs, device=DEVICE)
    fit_time = perf_counter() - start
    final = history.iloc[-1]
    result = {
        "model": model_name,
        "accuracy": final["test_accuracy"],
        "f1_macro": final["test_f1_macro"],
        "fit_time_sec": fit_time,
    }
    return model, history, result

scratch_cnn, scratch_history, scratch_result = train_variant(
    "simple_cnn_from_scratch",
    SimpleCNN(),
    simple_train_loader,
    simple_test_loader,
    lr=0.001,
    epochs=EPOCHS_SIMPLE,
)

frozen_mobilenet, frozen_history, frozen_result = train_variant(
    "mobilenetv2_frozen_features",
    build_finetune_model(n_classes=10, freeze_features=True),
    pretrained_train_loader,
    pretrained_test_loader,
    lr=0.001,
    epochs=EPOCHS_FINETUNE,
)

model_results = pd.DataFrame([scratch_result, frozen_result]).sort_values("f1_macro", ascending=False).reset_index(drop=True)
model_results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(scratch_history["epoch"], scratch_history["test_accuracy"], label="SimpleCNN")
axes[0].plot(frozen_history["epoch"], frozen_history["test_accuracy"], label="MobileNetV2 frozen")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("test accuracy")
axes[0].set_title("Accuracy curves")
axes[0].legend()

axes[1].plot(scratch_history["epoch"], scratch_history["test_loss"], label="SimpleCNN")
axes[1].plot(frozen_history["epoch"], frozen_history["test_loss"], label="MobileNetV2 frozen")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("test loss")
axes[1].set_title("Loss curves")
axes[1].legend()

plt.tight_layout()
plt.show()


## Guided Analysis: Confusion Matrix

Inspect the best model's mistakes.


In [ ]:
best_name = model_results.iloc[0]["model"]
if best_name == "simple_cnn_from_scratch":
    best_model = scratch_cnn
    eval_loader = simple_test_loader
else:
    best_model = frozen_mobilenet
    eval_loader = pretrained_test_loader

best_model.eval()
all_preds = []
all_targets = []
with torch.no_grad():
    for xb, yb in eval_loader:
        logits = best_model(xb.to(DEVICE))
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(yb.numpy())

pred = np.concatenate(all_preds)
true = np.concatenate(all_targets)
cm = confusion_matrix(true, pred)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.show()


## Task 5 - Benchmark Long Format

Implement `make_benchmark_long(results, week, dataset, task_type, target, split)`.

Convert model results into long benchmark format:

- `week`
- `dataset`
- `task_type`
- `target`
- `model`
- `metric`
- `score`
- `split`
- `notes`

Include numeric metric columns such as `accuracy` and `f1_macro`. Do not include `fit_time_sec` as a metric.


In [ ]:
def make_benchmark_long(
    results: pd.DataFrame,
    week: str,
    dataset: str,
    task_type: str,
    target: str,
    split: str,
) -> pd.DataFrame:
    """Convert image-classification results to the cumulative benchmark long format."""
    metric_columns = [column for column in results.columns if column not in ["model", "fit_time_sec"]]
    rows = []

    for _, result in results.iterrows():
        for metric in metric_columns:
            if pd.isna(result[metric]):
                continue
            rows.append(
                {
                    "week": week,
                    "dataset": dataset,
                    "task_type": task_type,
                    "target": target,
                    "model": result["model"],
                    "metric": metric,
                    "score": float(result[metric]),
                    "split": split,
                    "notes": "balanced CIFAR-10 subset; scratch CNN and pretrained MobileNetV2",
                }
            )

    columns = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
    return pd.DataFrame(rows, columns=columns)


In [ ]:
# Self-check: Task 5
benchmark_long = make_benchmark_long(
    model_results,
    week="W09",
    dataset="CIFAR-10 subset",
    task_type="image_classification",
    target="object_class",
    split=f"balanced_{TRAIN_PER_CLASS}_train_{TEST_PER_CLASS}_test_per_class",
)

expected_cols = ["week", "dataset", "task_type", "target", "model", "metric", "score", "split", "notes"]
assert list(benchmark_long.columns) == expected_cols
assert {"accuracy", "f1_macro"}.issubset(set(benchmark_long["metric"]))
assert "fit_time_sec" not in set(benchmark_long["metric"])
assert benchmark_long["score"].between(0, 1).all()

print("Task 5 passed")
benchmark_long.head(10)


## Benchmark Wide View

This is the key comparison: training from scratch vs adapting a pretrained visual model.


In [ ]:
benchmark_wide = (
    benchmark_long
    .pivot_table(
        index=["dataset", "task_type", "target", "metric", "split"],
        columns="model",
        values="score",
        aggfunc="first",
    )
    .reset_index()
)
benchmark_wide.columns.name = None
benchmark_wide


## Reflection

Answer briefly, but concretely.

1. Did the frozen pretrained MobileNetV2 outperform the simple CNN from scratch? By how much?
2. What did the pretrained model bring to the problem before seeing CIFAR-10?
3. What are the trade-offs of resizing 32-by-32 images to 224-by-224 for transfer learning?
4. If you had more time or compute, would you unfreeze more layers? Why?


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - Unfreeze the Last Block
Unfreeze the last MobileNetV2 feature block and train for 1-2 more epochs with a smaller learning rate. Add the result to the benchmark.

### Track B - More Training Data
Increase `TRAIN_PER_CLASS` and see whether the scratch CNN narrows the gap.

### Track C - Try Another Pretrained Model
Replace MobileNetV2 with ResNet18 or EfficientNet-B0. Keep the benchmark format unchanged.


In [ ]:
# Optional challenge workspace
# Your code here
